In [11]:
!pip install PySastrawi


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# =========================================================
# CELL 1 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import sys
import pandas as pd
import numpy as np

from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

BACKEND_DIR = next(
    path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "backend")
    if (path / "src" / "preprocessing").is_dir()
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.preprocessing.stopwords import get_stopwords

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


print("✅ Import berhasil")
print("✅ Supabase client siap") 
# =========================================================
# NAMA TABEL SUPABASE
# =========================================================
SOURCE_TABLE = "scholar_article_doi"
CLEANED_TABLE = "cleaned_papers_results"
EVIDENCE_TABLE = "preprocessing_evidence"

print("✅ Nama tabel siap")
print("Source:", SOURCE_TABLE)
print("Cleaned:", CLEANED_TABLE)
print("Evidence:", EVIDENCE_TABLE)

✅ Import berhasil
✅ Supabase client siap
✅ Nama tabel siap
Source: scholar_article_doi
Cleaned: cleaned_papers_results
Evidence: preprocessing_evidence


In [13]:
# =========================================================
# LOAD DATA DARI SUPABASE DOI (AUTO BATCH)
# =========================================================
print("📥 Mengambil data dari Supabase...")

SOURCE_TABLE = "scholar_article_doi"

all_data = []
batch_size = 1000
offset = 0

selected_columns = """
id,
title,
abstract,
authors,
source,
year,
url,
scraped_at,
scrape_status,
pdf_url,
category,
doi,
pdf_doi,
access_type
"""

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select(selected_columns)
        .range(offset, offset + batch_size - 1)
        .execute()
    )

    batch = response.data or []
    if not batch:
        break

    all_data.extend(batch)
    print(f"  → Batch {offset // batch_size + 1}: {len(batch)} data")

    if len(batch) < batch_size:
        break

    offset += batch_size

df = pd.DataFrame(all_data)

print(f"✅ Total data dari {SOURCE_TABLE}: {len(df)} baris")
df.head(3)

📥 Mengambil data dari Supabase...
  → Batch 1: 200 data
✅ Total data dari scholar_article_doi: 200 baris


,id,title,abstract,authors,source,year,url,scraped_at,scrape_status,pdf_url,category,doi,pdf_doi,access_type
0,1,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,Machine learning and the city: applications in …,2022,https://onlinelibrary.wiley.com/doi/abs/10.100...,2026-07-11T12:02:19.261489+00:00,metadata_only,None,machine learning,10.1002/9781119815075.ch18,None,closed_access
1,2,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",Nature Reviews …,2024,https://www.nature.com/articles/s41579-023-009...,2026-07-11T12:02:43.52782+00:00,metadata_only,https://www.nature.com/articles/s41579-023-009...,machine learning,10.1038/s41579-023-00984-1,None,closed_access
2,3,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",… of the AAAI conference on artificial …,2021,https://ojs.aaai.org/index.php/AAAI/article/vi...,2026-07-11T12:03:04.060467+00:00,pdf_downloaded,https://ojs.aaai.org/index.php/AAAI/article/do...,machine learning,10.1609/aaai.v35i13.17371,None,open_access


In [14]:
# =========================================================
# CELL 3 - VALIDASI KOLOM WAJIB
# =========================================================
required_cols = [
    "id",
    "title",
    "abstract",
    "authors",
    "source",
    "year",
    "url",
    "scraped_at",
    "scrape_status",
    "pdf_url",
    "category",
    "doi",
    "pdf_doi",
    "access_type"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

for col in required_cols:
    df[col] = df[col].fillna("")

df["id"] = df["id"].astype("int64")

print("✅ Kolom wajib valid")
df[required_cols].head(3)

✅ Kolom wajib valid


,id,title,abstract,authors,source,year,url,scraped_at,scrape_status,pdf_url,category,doi,pdf_doi,access_type
0,1,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,Machine learning and the city: applications in …,2022,https://onlinelibrary.wiley.com/doi/abs/10.100...,2026-07-11T12:02:19.261489+00:00,metadata_only,,machine learning,10.1002/9781119815075.ch18,,closed_access
1,2,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",Nature Reviews …,2024,https://www.nature.com/articles/s41579-023-009...,2026-07-11T12:02:43.52782+00:00,metadata_only,https://www.nature.com/articles/s41579-023-009...,machine learning,10.1038/s41579-023-00984-1,,closed_access
2,3,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",… of the AAAI conference on artificial …,2021,https://ojs.aaai.org/index.php/AAAI/article/vi...,2026-07-11T12:03:04.060467+00:00,pdf_downloaded,https://ojs.aaai.org/index.php/AAAI/article/do...,machine learning,10.1609/aaai.v35i13.17371,,open_access


In [15]:
# =========================================================
# CELL 4 - STOPWORDS + STEMMER
# =========================================================
stop_words = get_stopwords()

stemmer = StemmerFactory().create_stemmer()

print("✅ Stopwords dan stemmer siap")

✅ Stopwords dan stemmer siap


In [16]:
# =========================================================
# CELL 5 - FUNGSI PREPROCESSING
# =========================================================
def safe_text(value):
    if value is None:
        return ""

    if isinstance(value, list):
        return " ".join(str(item) for item in value)

    if isinstance(value, dict):
        return " ".join(str(item) for item in value.values())

    return str(value)

def clean_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"doi\s*[:/]?\s*", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenizing(text):
    if not isinstance(text, str):
        return []
    return text.split()

def remove_stopwords(tokens):
    if not isinstance(tokens, list):
        return []

    return [
        token for token in tokens
        if token not in stop_words and len(token) > 1
    ]

def stemming(tokens):
    if not isinstance(tokens, list):
        return []

    return [stemmer.stem(token) for token in tokens]

print("✅ Fungsi preprocessing siap")

✅ Fungsi preprocessing siap


In [17]:
# =========================================================
# CELL 6 - GABUNGKAN SEMUA METADATA MENJADI FULL_TEXT
# =========================================================
metadata_cols = [
    "title",
    "abstract",
    "authors",
    "source",
    "year",    
    "url",
    "scraped_at",
    "scrape_status",
    "pdf_url",
    "category",
    "doi",
    "pdf_doi",
    "access_type"
]

df["full_text"] = df[metadata_cols].astype(str).agg(" ".join, axis=1).str.strip()

print("✅ full_text dibuat dari semua metadata")
df[["id", "title", "full_text"]].head(3)

✅ full_text dibuat dari semua metadata


,id,title,full_text
0,1,What is machine learning?,What is machine learning? … that one can emplo...
1,2,Machine learning for microbiologists,Machine learning for microbiologists … how to ...
2,3,Amnesiac machine learning,Amnesiac machine learning … It gives EU reside...


In [18]:
# =========================================================
# CELL 7 - STEP 1 TEXT CLEANSING
# =========================================================
df["cleaned"] = df["full_text"].apply(clean_text)

print("✅ Step 1 selesai: Text Cleansing")
df[["full_text", "cleaned"]].head(3)

✅ Step 1 selesai: Text Cleansing


,full_text,cleaned
0,What is machine learning? … that one can emplo...,what is machine learning that one can employ i...
1,Machine learning for microbiologists … how to ...,machine learning for microbiologists how to ev...
2,Amnesiac machine learning … It gives EU reside...,amnesiac machine learning it gives eu resident...


In [19]:
# =========================================================
# CELL 8 - STEP 2 TOKENIZATION
# =========================================================
df["tokens"] = df["cleaned"].apply(tokenizing)

print("✅ Step 2 selesai: Tokenization")
df[["cleaned", "tokens"]].head(3)

✅ Step 2 selesai: Tokenization


,cleaned,tokens
0,what is machine learning that one can employ i...,"[what, is, machine, learning, that, one, can, ..."
1,machine learning for microbiologists how to ev...,"[machine, learning, for, microbiologists, how,..."
2,amnesiac machine learning it gives eu resident...,"[amnesiac, machine, learning, it, gives, eu, r..."


In [20]:
# =========================================================
# CELL 9 - STEP 3 STOPWORD REMOVAL
# =========================================================
df["tokens_clean"] = df["tokens"].apply(remove_stopwords)

print("✅ Step 3 selesai: Stopword Removal")
df[["tokens", "tokens_clean"]].head(3)

✅ Step 3 selesai: Stopword Removal


,tokens,tokens_clean
0,"[what, is, machine, learning, that, one, can, ...","[what, machine, learning, one, can, employ, ma..."
1,"[machine, learning, for, microbiologists, how,...","[machine, learning, microbiologists, how, eval..."
2,"[amnesiac, machine, learning, it, gives, eu, r...","[amnesiac, machine, learning, gives, eu, resid..."


In [21]:
# =========================================================
# STEP 10 - STEMMING (Sastrawi / Nazief-Adriani)
# =========================================================
df["tokens_stemmed"] = df["tokens_clean"].apply(stemming)

print("✅ Step 4 selesai: Stemming")
df[["tokens_clean", "tokens_stemmed"]].head(3)

✅ Step 4 selesai: Stemming


,tokens_clean,tokens_stemmed
0,"[what, machine, learning, one, can, employ, ma...","[what, machine, learning, one, can, employ, ma..."
1,"[machine, learning, microbiologists, how, eval...","[machine, learning, microbiologists, how, eval..."
2,"[amnesiac, machine, learning, gives, eu, resid...","[amnesiac, machine, learning, gives, eu, resid..."


In [22]:
# =========================================================
# CELL 11 - GABUNGKAN TOKEN MENJADI CLEANED_TEXT
# =========================================================
df["cleaned_text"] = df["tokens_stemmed"].apply(
    lambda tokens: " ".join(tokens) if isinstance(tokens, list) else ""
)

print("✅ cleaned_text selesai dibuat")
print(f"📊 Total baris diproses: {len(df)}")
print(f"📊 cleaned_text kosong: {int(df['cleaned_text'].eq('').sum())}")

df[["title", "cleaned_text"]].head(5)

✅ cleaned_text selesai dibuat
📊 Total baris diproses: 200
📊 cleaned_text kosong: 0


,title,cleaned_text
0,What is machine learning?,what machine learning one can employ machine l...
1,Machine learning for microbiologists,machine learning microbiologists how evaluate ...
2,Amnesiac machine learning,amnesiac machine learning gives eu residents a...
3,Designing nanotheranostics with machine learning,designing nanotheranostics machine learning ke...
4,A guide to machine learning for biologists,guide machine learning biologists machine lear...


In [23]:
# =========================================================
# AUDIT KUALITAS PREPROCESSING
# =========================================================
df["len_full_text"] = df["full_text"].apply(lambda x: len(str(x).split()))
df["len_cleaned_text"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))

print("Rata-rata token sebelum preprocessing:", round(df["len_full_text"].mean(), 2))
print("Rata-rata token sesudah preprocessing:", round(df["len_cleaned_text"].mean(), 2))
print("Dokumen jadi kosong setelah preprocessing:", int((df["len_cleaned_text"] == 0).sum()))



Rata-rata token sebelum preprocessing: 57.04
Rata-rata token sesudah preprocessing: 54.4
Dokumen jadi kosong setelah preprocessing: 0


In [24]:
# =========================================================
# CELL 13 - CONTOH DATA ACAK
# =========================================================
sample_cols = [
    "title",
    "full_text",
    "cleaned",
    "tokens",
    "tokens_clean",
    "tokens_stemmed",
    "cleaned_text"
]

df[sample_cols].sample(min(5, len(df)), random_state=42)

,title,full_text,cleaned,tokens,tokens_clean,tokens_stemmed,cleaned_text
95,Digitalisasi Usaha Mikro Kecil Dan Menengah Di...,Digitalisasi Usaha Mikro Kecil Dan Menengah Di...,digitalisasi usaha mikro kecil dan menengah di...,"[digitalisasi, usaha, mikro, kecil, dan, menen...","[digitalisasi, usaha, mikro, kecil, menengah, ...","[digitalisasi, usaha, mikro, kecil, tengah, de...",digitalisasi usaha mikro kecil tengah desa lal...
15,Research on machine learning with algorithms a...,Research on machine learning with algorithms a...,research on machine learning with algorithms a...,"[research, on, machine, learning, with, algori...","[research, machine, learning, algorithms, deve...","[research, machine, learning, algorithms, deve...",research machine learning algorithms developme...
30,Penerapan Machine Learning dan Deep Learning p...,Penerapan Machine Learning dan Deep Learning p...,penerapan machine learning dan deep learning p...,"[penerapan, machine, learning, dan, deep, lear...","[penerapan, machine, learning, deep, learning,...","[terap, machine, learning, deep, learning, tin...",terap machine learning deep learning tingkat d...
158,Cyber governance studies in ensuring cybersecu...,Cyber governance studies in ensuring cybersecu...,cyber governance studies in ensuring cybersecu...,"[cyber, governance, studies, in, ensuring, cyb...","[cyber, governance, studies, ensuring, cyberse...","[cyber, governance, studies, ensuring, cyberse...",cyber governance studies ensuring cybersecurit...
128,Implemetasi aplikasi mobile pengenalan kampus ...,Implemetasi aplikasi mobile pengenalan kampus ...,implemetasi aplikasi mobile pengenalan kampus ...,"[implemetasi, aplikasi, mobile, pengenalan, ka...","[implemetasi, aplikasi, mobile, pengenalan, ka...","[implemetasi, aplikasi, mobile, kenal, kampus,...",implemetasi aplikasi mobile kenal kampus masa ...


In [25]:
# =========================================================
# CELL 14 - HELPER UPSERT SUPABASE
# =========================================================
def to_records_safe(dataframe: pd.DataFrame):
    safe_df = dataframe.replace({np.nan: None})
    return safe_df.to_dict(orient="records")

def upsert_batches(table_name: str, records: list, on_conflict: str, batch_size: int = 500):
    total = len(records)

    if total == 0:
        print(f"⚠️ Tidak ada data untuk di-upsert ke {table_name}")
        return

    for i in range(0, total, batch_size):
        batch = records[i:i + batch_size]

        supabase.table(table_name).upsert(
            batch,
            on_conflict=on_conflict
        ).execute()

        print(f"  → Batch {i // batch_size + 1}: {len(batch)} data")

    print(f"✅ Upsert {total} baris ke {table_name}")

In [26]:
# =========================================================
# IMPORT TAMBAHAN UNTUK TIMESTAMP
# =========================================================
from datetime import datetime, timezone

ts = datetime.now(timezone.utc).isoformat()
print("Timestamp:", ts)
# =========================================================
# CELL 16 - SIMPAN HASIL CLEANED (CSV + SUPABASE)
# target table: cleaned_papers_results
# =========================================================
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
save_dir = os.path.join(base_dir, "data")
save_path = os.path.join(save_dir, "cleaned_papers.csv")
os.makedirs(save_dir, exist_ok=True)

save_df = df[
    ["id", "title", "abstract", "authors", "year", "source", "category", "pdf_url", "url", "scrape_status", "cleaned_text"]
].copy()

save_df["updated_at"] = ts
save_df["id"] = save_df["id"].astype("int64")

save_df.to_csv(save_path, index=False)

print(f"✅ CSV cleaned diperbarui: {save_path}")
print(f"📊 Total baris CSV: {len(save_df)}")

upsert_batches(
    table_name=CLEANED_TABLE,
    records=to_records_safe(save_df),
    on_conflict="id",
    batch_size=500
)

save_df.head(5)

Timestamp: 2026-07-11T16:46:23.070865+00:00
✅ CSV cleaned diperbarui: d:\Tugas Akhir\paperci_artikel\backend\data\cleaned_papers.csv
📊 Total baris CSV: 200
  → Batch 1: 200 data
✅ Upsert 200 baris ke cleaned_papers_results


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status,cleaned_text,updated_at
0,1,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,,https://onlinelibrary.wiley.com/doi/abs/10.100...,metadata_only,what machine learning one can employ machine l...,2026-07-11T16:46:23.070865+00:00
1,2,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,https://www.nature.com/articles/s41579-023-009...,https://www.nature.com/articles/s41579-023-009...,metadata_only,machine learning microbiologists how evaluate ...,2026-07-11T16:46:23.070865+00:00
2,3,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",2021,… of the AAAI conference on artificial …,machine learning,https://ojs.aaai.org/index.php/AAAI/article/do...,https://ojs.aaai.org/index.php/AAAI/article/vi...,pdf_downloaded,amnesiac machine learning gives eu residents a...,2026-07-11T16:46:23.070865+00:00
3,4,Designing nanotheranostics with machine learning,"… As a key branch of artificial intelligence, ...","L Rao, Y Yuan, X Shen, G Yu, X Chen",2024,Nature Nanotechnology,machine learning,https://www.nature.com/articles/s41565-024-017...,https://www.nature.com/articles/s41565-024-017...,metadata_only,designing nanotheranostics machine learning ke...,2026-07-11T16:46:23.070865+00:00
4,5,A guide to machine learning for biologists,… A machine learning task is an objective spec...,"JG Greener, SM Kandathil, L Moffat…",2022,Nature reviews Molecular …,machine learning,https://www.nature.com/articles/s41580-021-004...,https://www.nature.com/articles/s41580-021-004...,metadata_only,guide machine learning biologists machine lear...,2026-07-11T16:46:23.070865+00:00


In [27]:
# =========================================================
# CELL 17 - SIMPAN EVIDENCE PREPROCESSING
# target table: preprocessing_evidence
# =========================================================
evidence_path = os.path.join(save_dir, "preprocessing_evidence_doi_sample.csv")

evidence_cols = [
    "id",
    "title",
    "full_text",
    "cleaned",
    "tokens",
    "tokens_clean",
    "tokens_stemmed",
    "cleaned_text"
]

evidence_df = df[evidence_cols].copy()

evidence_df["updated_at"] = ts
evidence_df["id"] = evidence_df["id"].astype("int64")

evidence_df.head(50).to_csv(evidence_path, index=False)

print(f"✅ Evidence sample CSV diperbarui: {evidence_path}")

upsert_batches(
    table_name=EVIDENCE_TABLE,
    records=to_records_safe(evidence_df),
    on_conflict="id",
    batch_size=500
)

print(f"📊 Total baris evidence ke Supabase: {len(evidence_df)}")
evidence_df.head(5)

✅ Evidence sample CSV diperbarui: d:\Tugas Akhir\paperci_artikel\backend\data\preprocessing_evidence_doi_sample.csv
  → Batch 1: 200 data
✅ Upsert 200 baris ke preprocessing_evidence
📊 Total baris evidence ke Supabase: 200


,id,title,full_text,cleaned,tokens,tokens_clean,tokens_stemmed,cleaned_text,updated_at
0,1,What is machine learning?,What is machine learning? … that one can emplo...,what is machine learning that one can employ i...,"[what, is, machine, learning, that, one, can, ...","[what, machine, learning, one, can, employ, ma...","[what, machine, learning, one, can, employ, ma...",what machine learning one can employ machine l...,2026-07-11T16:46:23.070865+00:00
1,2,Machine learning for microbiologists,Machine learning for microbiologists … how to ...,machine learning for microbiologists how to ev...,"[machine, learning, for, microbiologists, how,...","[machine, learning, microbiologists, how, eval...","[machine, learning, microbiologists, how, eval...",machine learning microbiologists how evaluate ...,2026-07-11T16:46:23.070865+00:00
2,3,Amnesiac machine learning,Amnesiac machine learning … It gives EU reside...,amnesiac machine learning it gives eu resident...,"[amnesiac, machine, learning, it, gives, eu, r...","[amnesiac, machine, learning, gives, eu, resid...","[amnesiac, machine, learning, gives, eu, resid...",amnesiac machine learning gives eu residents a...,2026-07-11T16:46:23.070865+00:00
3,4,Designing nanotheranostics with machine learning,Designing nanotheranostics with machine learni...,designing nanotheranostics with machine learni...,"[designing, nanotheranostics, with, machine, l...","[designing, nanotheranostics, machine, learnin...","[designing, nanotheranostics, machine, learnin...",designing nanotheranostics machine learning ke...,2026-07-11T16:46:23.070865+00:00
4,5,A guide to machine learning for biologists,A guide to machine learning for biologists … A...,a guide to machine learning for biologists a m...,"[a, guide, to, machine, learning, for, biologi...","[guide, machine, learning, biologists, machine...","[guide, machine, learning, biologists, machine...",guide machine learning biologists machine lear...,2026-07-11T16:46:23.070865+00:00


In [28]:
# =========================================================
# CELL 18 - VALIDASI CEPAT SUPABASE
# =========================================================
check_cleaned = (
    supabase.table(CLEANED_TABLE)
    .select("id", count="exact")
    .limit(1)
    .execute()
)

check_evidence = (
    supabase.table(EVIDENCE_TABLE)
    .select("id", count="exact")
    .limit(1)
    .execute()
)

print("✅ cleaned_papers_results count:", check_cleaned.count)
print("✅ preprocessing_evidence count:", check_evidence.count)

✅ cleaned_papers_results count: 200
✅ preprocessing_evidence count: 200
